In [0]:
data = [(1, 100), (2, 200), (3, 300)]
schema = "id INT, salary INT"

In [0]:
df = spark.createDataFrame(data, schema)
display(df)

In [0]:
#Here I clicked the run button 5 times, This creates the 5 parquet files in the delta log.
df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/")

In [0]:
%sql
SELECT * FROM delta.`/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/`

**OPTIMIZE:**
 _Optimize is a performance improvement command used on delta tables into:_
- Compact small files into larger efficient files.
- Remove deleted files.
- Improve Read Performance
- Improve Zordering for selective queries


**ZORDER**
- _Zorder improves data skipping by collecting similar values. It is recommended when queries filter frequently on a specific column_

In [0]:
%sql
OPTIMIZE delta.`/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/` ZORDER BY id

### **LIQUID CLUSTERING**
_Liquid Clustering is an advanced Delta Lake feature that automatically organize and optimize data layout without needing partitioning_

**Creating a liquid table on bronze layer**

In [0]:
%sql
CREATE OR REPLACE TABLE databricksgovi.bronze.liquid_table
(
  id INT,
  salary INT
)
CLUSTER BY (id)

In [0]:
%sql
ALTER TABLE databricksgovi.bronze.liquid_table
CLUSTER BY none

**If we not sure on to apply the CLUSTER BY on which column**

_Databricks have a feature called AUTO_

In [0]:
%sql
CREATE OR REPLACE TABLE databricksgovi.bronze.liquid_table2
(
  id INT,
  salary INT
)
CLUSTER BY AUTO

### **DATA VERSIONING**
_Data Versioning means keeping track of every change made to data in the data lake, so we can travel back to old versions of the data at any time_

In [0]:
%sql
SELECT * FROM delta.`/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/`

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/`

### **DATA TIME TRAVEL USING DELTA**
_Time Travel in delta lake allows you to query, restore or read data from an easier version of a delta lake table_

In [0]:
%sql
DELETE FROM delta.`/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/`

In [0]:
%sql
Select * from delta.`/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/`

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/`

**Restoring the data**

In [0]:
%sql
RESTORE delta.`/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/` TO VERSION AS OF 5

In [0]:
%sql
select * from delta.`/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/`

### VACUUM COMMAND
_Use This command with precaution why because one the files deleted, never recovered again_

In [0]:
%sql
VACUUM delta.`/Volumes/databricksgovi/bronze/bronze_volume/delta_optimize/` RETAIN 0 HOURS DRY RUN